# Final Stage F: Direct 18-Mask Coordinate Comparator

**Runtime:** CPU. Select **Runtime > Run all**.

This version **never indexes or recursively scans the 51R2/53R2 capture directories**. It consumes the previous seven-page comparator's `call_signature_differences.csv`, reads only the 18 exceptional call records, resolves only those 18 `mask.npy` files, stages those masks to Colab local storage, then compares coordinates and Notebook45 regional ownership.

If `call_signature_differences.csv` does not contain resolvable call paths, the notebook stops and tells you exactly which small override table is needed. It does not fall back to a full capture scan.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scipy


In [ ]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# SET THIS to the seven-page comparator output directory
# ------------------------------------------------------------

SEVEN_PAGE_RUN = Path(
    '/content/drive/MyDrive/MusicAnnotationProject/'
    'stage_f_final_call_inputs/comparison_20261104_154845'
)

files = {
    '51R2': SEVEN_PAGE_RUN / 'tables/unmatched_51r2_calls.csv',
    '53R2': SEVEN_PAGE_RUN / 'tables/unmatched_53r2_calls.csv',
}

FAILING_PAGES = {
    'Page003',
    'Page004',
    'Page005',
    'Page008',
    'Page036',
    'Page040',
    'Page049',
}

rows = []

for side, csv_path in files.items():

    if not csv_path.exists():
        print('MISSING:', csv_path)
        continue

    df = pd.read_csv(csv_path)

    print('\n', '=' * 70)
    print(side)
    print('File:', csv_path)
    print('Columns:', df.columns.tolist())
    print('=' * 70)

    # Find likely page column.
    page_col = next(
        (
            c for c in df.columns
            if c.lower() in {
                'page',
                'page_51',
                'page_53',
            }
        ),
        None,
    )

    # Find likely call-directory/path column.
    path_candidates = [
        c for c in df.columns
        if any(
            token in c.lower()
            for token in (
                'call',
                'path',
                'dir',
            )
        )
    ]

    print('Likely page column:', page_col)
    print('Likely path columns:', path_candidates)

    if page_col is not None:
        df = df[
            df[page_col]
            .astype(str)
            .isin(FAILING_PAGES)
        ].copy()

    display(df)

    # Look at each plausible path column.
    for path_col in path_candidates:

        for _, record in df.iterrows():

            raw = str(record[path_col]).strip()

            if (
                not raw
                or raw.lower() == 'nan'
            ):
                continue

            p = Path(raw)

            # Record may already point to mask.npy.
            if p.name == 'mask.npy':
                mask_path = p

            # More likely it points to the call directory.
            elif p.is_dir():
                mask_path = p / 'mask.npy'

            else:
                # Still construct the expected path so we can
                # see whether it exists.
                mask_path = p / 'mask.npy'

            if mask_path.exists():

                page = (
                    str(record[page_col])
                    if page_col is not None
                    else None
                )

                rows.append({
                    'side': side,
                    'page': page,
                    'mask_path': str(mask_path),
                    'source_csv': str(csv_path),
                    'source_column': path_col,
                })


resolved = pd.DataFrame(rows).drop_duplicates(
    ['side', 'page', 'mask_path']
)

print('\n' + '=' * 80)
print('RESOLVED EXCEPTIONAL MASK PATHS')
print('=' * 80)

display(
    resolved.sort_values(
        ['page', 'side', 'mask_path']
    )
)

print('\nCounts by side/page:')

display(
    resolved.groupby(
        ['side', 'page']
    ).size().reset_index(
        name='masks'
    )
)

output = (
    SEVEN_PAGE_RUN /
    'exceptional_call_paths_resolved.csv'
)

resolved.to_csv(
    output,
    index=False,
)

print('\nSaved:')
print(output)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, json, platform, re, shutil, sys

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Optional exact previous-comparator run. Leave None for small-output discovery.
PREVIOUS_COMPARATOR_RUN_OVERRIDE = None

# Optional exact persisted Notebook45 run/root containing masks/<Page>/<region>/M3.png.
NOTEBOOK45_ROOT_OVERRIDE = None

# Optional manual path table. Used only if the previous comparator table lacks
# actual call/mask paths. Expected columns are documented below.
EXCEPTIONAL_CALL_PATHS_OVERRIDE = None

FAILING_PAGES = {
    'Page003', 'Page004', 'Page005', 'Page008',
    'Page036', 'Page040', 'Page049',
}
EXPECTED_PER_SIDE = 9
EXPECTED_PAIRS = 9

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_direct_18_mask_comparison'
OUT = ROOT / 'stage_f_direct_18_mask_coordinate_comparison' / RUN_ID
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
IMAGES = OUT / 'images'
LOCAL = Path('/content/stage_f_direct_18_masks')
for d in [OUT, TABLES, AUDIT, IMAGES]:
    d.mkdir(parents=True, exist_ok=False)
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True, exist_ok=False)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def semantic_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def page_of(value):
    m = re.search(r'Page\d+', str(value), re.I)
    return None if not m else 'Page' + m.group(0)[4:].zfill(3)


def normalise_side(value):
    text = str(value).upper().replace('_','')
    if '51R2' in text or '51' == text:
        return '51R2'
    if '53R2' in text or '53' == text:
        return '53R2'
    return None

# ------------------------------------------------------------------
# 1. Locate ONLY the previous seven-page comparator summary.
#    We intentionally search for one filename, not call directories.
# ------------------------------------------------------------------

def candidate_previous_runs():
    names = {'call_signature_differences.csv', 'call_population_by_page.csv'}
    roots = {}
    for filename in names:
        for p in ROOT.rglob(filename):
            # Small result-table search only. Never descend into calls/ here.
            if '/calls/' in str(p).replace('\\','/'):
                continue
            roots.setdefault(p.parent, set()).add(filename)
    hits = []
    for table_dir, found in roots.items():
        if 'call_signature_differences.csv' in found:
            run_root = table_dir.parent if table_dir.name == 'tables' else table_dir
            hits.append(run_root)
    return sorted(set(hits), key=lambda p: p.stat().st_mtime)

if PREVIOUS_COMPARATOR_RUN_OVERRIDE is not None:
    PREV = Path(PREVIOUS_COMPARATOR_RUN_OVERRIDE)
else:
    hits = candidate_previous_runs()
    print('Previous comparator candidates:')
    for x in hits[-10:]:
        print(' ', x)
    require(hits, 'No previous seven-page comparator run with call_signature_differences.csv was found. Set PREVIOUS_COMPARATOR_RUN_OVERRIDE.')
    PREV = hits[-1]

sig_path = PREV / 'call_signature_differences.csv'
if not sig_path.exists():
    sig_path = PREV / 'tables' / 'call_signature_differences.csv'
require(sig_path.exists(), f'Missing call_signature_differences.csv below {PREV}')
sig = pd.read_csv(sig_path, dtype=str, keep_default_na=False)
print('Previous comparator:', PREV)
print('Signature table:', sig_path)
print('Rows:', len(sig), 'Columns:', sig.columns.tolist())

# ------------------------------------------------------------------
# 2. Extract only ONLY_51R2 / ONLY_53R2 rows.
# ------------------------------------------------------------------
status_col = next((c for c in sig.columns if c.lower() in {'status','side','difference_status'}), None)
require(status_col is not None, f'Cannot identify status column in {sig.columns.tolist()}')
page_col = next((c for c in sig.columns if c.lower() == 'page'), None)
require(page_col is not None, 'call_signature_differences.csv has no page column.')

exceptional = sig[sig[status_col].astype(str).str.upper().isin({'ONLY_51R2','ONLY_53R2'})].copy()
exceptional['page_norm'] = exceptional[page_col].map(page_of)
exceptional = exceptional[exceptional.page_norm.isin(FAILING_PAGES)].copy()
print('Exceptional signature rows:', len(exceptional))
print(exceptional.groupby([status_col,'page_norm']).size())

# ------------------------------------------------------------------
# 3. Resolve actual 18 mask paths WITHOUT scanning capture roots.
# ------------------------------------------------------------------
path_columns = [c for c in exceptional.columns if any(k in c.lower() for k in ['mask_path','call_dir','call_path','metadata_path','directory','path'])]
call_columns = [c for c in exceptional.columns if 'call' in c.lower() and c not in path_columns]


def existing_mask_from_row(row):
    candidates = []
    for c in path_columns:
        raw = str(row.get(c,'')).strip()
        if not raw:
            continue
        p = Path(raw)
        if p.name == 'mask.npy':
            candidates.append(p)
        elif p.suffix.lower() == '.json':
            candidates.append(p.parent / 'mask.npy')
        else:
            candidates.append(p / 'mask.npy')
    existing = []
    for p in candidates:
        if p.exists() and p.is_file():
            rp = p.resolve()
            if rp not in existing:
                existing.append(rp)
    return existing

resolved_rows = []
unresolved_rows = []
for _, row in exceptional.iterrows():
    status = str(row[status_col]).upper()
    side = '51R2' if status == 'ONLY_51R2' else '53R2'
    masks = existing_mask_from_row(row)
    if len(masks) == 1:
        resolved_rows.append({'side':side,'page':row.page_norm,'mask_path':str(masks[0]),'source_row':int(row.name)})
    else:
        unresolved_rows.append({'side':side,'page':row.page_norm,'source_row':int(row.name),'resolved_candidates':len(masks)})

# Manual override is a deliberately tiny 18-row table if historical result
# paths were not written into the previous comparator table.
if unresolved_rows:
    if EXCEPTIONAL_CALL_PATHS_OVERRIDE is None:
        pd.DataFrame(unresolved_rows).to_csv(AUDIT/'unresolved_exceptional_rows.csv', index=False)
        template = pd.DataFrame(columns=['side','page','mask_path'])
        template.to_csv(AUDIT/'exceptional_call_paths_override_TEMPLATE.csv', index=False)
        raise RuntimeError(
            'The previous comparator identified the exceptional signatures but did not persist resolvable call/mask paths. '
            'This notebook will NOT scan the full capture directories. Fill the 18-row template at '
            f'{AUDIT/"exceptional_call_paths_override_TEMPLATE.csv"} with side,page,mask_path, set '
            'EXCEPTIONAL_CALL_PATHS_OVERRIDE to that CSV, and rerun.'
        )
    override = pd.read_csv(Path(EXCEPTIONAL_CALL_PATHS_OVERRIDE), dtype=str)
    require({'side','page','mask_path'}.issubset(override.columns), 'Override table must contain side,page,mask_path.')
    resolved_rows = []
    for _, r in override.iterrows():
        side = normalise_side(r.side); page = page_of(r.page); p = Path(r.mask_path)
        require(side in {'51R2','53R2'} and page in FAILING_PAGES and p.exists(), f'Invalid override row: {r.to_dict()}')
        resolved_rows.append({'side':side,'page':page,'mask_path':str(p.resolve()),'source_row':None})

resolved = pd.DataFrame(resolved_rows)
require(len(resolved) == 18, f'Expected exactly 18 mask paths; found {len(resolved)}.')
require((resolved.side=='51R2').sum()==EXPECTED_PER_SIDE and (resolved.side=='53R2').sum()==EXPECTED_PER_SIDE,
        f'Expected 9 masks per side; got {resolved.side.value_counts().to_dict()}')
require(resolved.mask_path.nunique()==18, 'Duplicate mask paths detected.')
resolved.to_csv(TABLES/'resolved_18_mask_paths.csv', index=False)
print('Resolved 18 mask paths directly. No capture directory index was performed.')

# ------------------------------------------------------------------
# 4. Stage ONLY those 18 masks from Drive to /content.
# ------------------------------------------------------------------
staged=[]
for i,r in resolved.reset_index(drop=True).iterrows():
    src=Path(r.mask_path)
    dst=LOCAL/f'{i:02d}_{r.side}_{r.page}_{src.parent.name}_mask.npy'
    shutil.copy2(src,dst)
    staged.append({**r.to_dict(),'local_mask_path':str(dst),'drive_bytes':src.stat().st_size})
staged=pd.DataFrame(staged)
staged.to_csv(TABLES/'staged_18_masks.csv',index=False)
print('Staged masks locally:', len(staged), 'bytes:', int(staged.drive_bytes.sum()))

# ------------------------------------------------------------------
# 5. Pair the 9 exceptional masks page-locally by minimum XOR.
# ------------------------------------------------------------------
def load_local(path):
    return np.asarray(np.load(path, mmap_mode='r', allow_pickle=False)>0)

pairs=[]; cost_rows=[]
for page in sorted(FAILING_PAGES):
    L=staged[(staged.page==page)&(staged.side=='51R2')].to_dict('records')
    R=staged[(staged.page==page)&(staged.side=='53R2')].to_dict('records')
    require(len(L)==len(R), f'{page}: unmatched exceptional populations {len(L)} vs {len(R)}')
    if not L: continue
    LM=[load_local(x['local_mask_path']) for x in L]; RM=[load_local(x['local_mask_path']) for x in R]
    require(len({m.shape for m in LM+RM})==1, f'{page}: shape mismatch among exceptional masks')
    C=np.empty((len(L),len(R)),dtype=np.int64)
    for i,a in enumerate(LM):
        for j,b in enumerate(RM):
            C[i,j]=np.count_nonzero(a!=b)
            cost_rows.append({'page':page,'left_path':L[i]['mask_path'],'right_path':R[j]['mask_path'],'left_pixels':int(a.sum()),'right_pixels':int(b.sum()),'xor_pixels':int(C[i,j])})
    ii,jj=linear_sum_assignment(C)
    for i,j in zip(ii,jj):
        pairs.append({'page':page,'left':L[int(i)],'right':R[int(j)],'m51':LM[int(i)],'m53':RM[int(j)],'xor':int(C[int(i),int(j)])})
require(len(pairs)==EXPECTED_PAIRS,f'Expected nine pairs; found {len(pairs)}')
pd.DataFrame(cost_rows).to_csv(TABLES/'pairing_costs.csv',index=False)
print('Paired exceptional masks:',len(pairs))

# ------------------------------------------------------------------
# 6. Resolve Notebook45 WITHOUT scanning the entire project for M3.png.
#    Search only known Notebook45 parent roots by shallow run completeness.
# ------------------------------------------------------------------
N45_PARENT = ROOT/'observed_print_classifier_v8_6_notebook45'


def n45_complete(run):
    return run.is_dir() and (run/'masks').exists()

if NOTEBOOK45_ROOT_OVERRIDE is not None:
    N45=Path(NOTEBOOK45_ROOT_OVERRIDE)
    require(n45_complete(N45), f'Notebook45 override does not contain masks/: {N45}')
else:
    require(N45_PARENT.exists(), f'Notebook45 parent missing: {N45_PARENT}. Set NOTEBOOK45_ROOT_OVERRIDE.')
    runs=[p for p in N45_PARENT.iterdir() if n45_complete(p)]
    require(runs, f'No Notebook45 run with masks/ found directly below {N45_PARENT}. Set NOTEBOOK45_ROOT_OVERRIDE.')
    # Prefer runs that have all seven page mask directories; no recursive file scan.
    qualified=[]
    for run in runs:
        found=sum(int((run/'masks'/page).exists()) for page in FAILING_PAGES)
        if found==len(FAILING_PAGES): qualified.append(run)
    require(qualified, 'No Notebook45 run has mask directories for all seven pages. Set NOTEBOOK45_ROOT_OVERRIDE.')
    N45=max(qualified,key=lambda p:p.stat().st_mtime)
print('Notebook45:',N45)

# Load only regional M3.png below the seven page directories.
regions=defaultdict(list); inv=[]
for page in sorted(FAILING_PAGES):
    page_dir=N45/'masks'/page
    require(page_dir.exists(), f'{page}: Notebook45 mask directory missing: {page_dir}')
    # One shallow region level. Avoid project-wide rglob.
    region_dirs=[p for p in page_dir.iterdir() if p.is_dir()]
    for rd in sorted(region_dirs):
        p=rd/'M3.png'
        if not p.exists(): continue
        a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
        require(a is not None,f'Unreadable {p}')
        m=a>0
        regions[page].append({'region_id':rd.name,'path':p,'mask':m})
        inv.append({'page':page,'region_id':rd.name,'path':str(p),'pixels':int(m.sum()),'sha256':semantic_hash(m.astype(np.uint8))})
    require(regions[page], f'{page}: no regional M3.png files under {page_dir}')
pd.DataFrame(inv).to_csv(AUDIT/'notebook45_regional_m3_inventory.csv',index=False)

# ------------------------------------------------------------------
# 7. Compare coordinates and ownership.
# ------------------------------------------------------------------
pair_rows=[]; coord_rows=[]; owner_rows=[]
for pid,pair in enumerate(pairs,1):
    page=pair['page']; a=pair['m51']; b=pair['m53']
    for r in regions[page]: require(r['mask'].shape==a.shape,f'{page}/{r["region_id"]}: shape mismatch')
    shared=a&b; o51=a&~b; o53=b&~a; xor=a^b
    label=f'{page}_pair{pid:02d}'
    d=IMAGES/label; d.mkdir()
    for name,m in [('mask_51',a),('mask_53',b),('shared',shared),('only_51',o51),('only_53',o53),('xor',xor)]:
        cv2.imwrite(str(d/f'{name}.png'),m.astype(np.uint8)*255)
    pair_rows.append({'pair_id':pid,'page':page,'mask_51':pair['left']['mask_path'],'mask_53':pair['right']['mask_path'],
        'pixels_51':int(a.sum()),'pixels_53':int(b.sum()),'shared':int(shared.sum()),'only_51':int(o51.sum()),'only_53':int(o53.sum()),'xor':int(xor.sum())})
    for direction,diff in [('ONLY_51',o51),('ONLY_53',o53)]:
        counts=Counter()
        ys,xs=np.where(diff)
        for y,x in zip(ys,xs):
            owners=sorted(r['region_id'] for r in regions[page] if r['mask'][y,x])
            text='|'.join(owners); counts[text]+=1
            coord_rows.append({'pair_id':pid,'page':page,'direction':direction,'y':int(y),'x':int(x),'owner_count':len(owners),'owner_ids':text,'single_owner':len(owners)==1})
        for owner_text,n in counts.items():
            owner_rows.append({'pair_id':pid,'page':page,'direction':direction,'owner_ids':owner_text,'owner_count':0 if not owner_text else len(owner_text.split('|')),'pixels':int(n)})
P=pd.DataFrame(pair_rows).sort_values(['page','pair_id']); C=pd.DataFrame(coord_rows); O=pd.DataFrame(owner_rows)
P.to_csv(TABLES/'exceptional_mask_pairs.csv',index=False); C.to_csv(TABLES/'exceptional_mask_coordinate_differences.csv',index=False); O.to_csv(TABLES/'exceptional_coordinate_ownership_summary.csv',index=False)
require(len(C)>0,'Exceptional masks have no coordinate differences.')

single=bool(C.single_owner.all()); zero=int((C.owner_count==0).sum()); multi=int((C.owner_count>1).sum())
s53=bool((P.only_51==0).all() and (P.only_53>0).all()); s51=bool((P.only_53==0).all() and (P.only_51>0).all())
if s53 and single:
    diagnosis='53R2_EXCEPTIONAL_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_51R2'; correction='Reproduce the 51R2 regional pixel-assignment rule and remove the ONLY_53 coordinates from the nine 53R2 calls.'
elif s51 and single:
    diagnosis='51R2_EXCEPTIONAL_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_53R2'; correction='Reproduce the 51R2 regional pixel-assignment rule and add the ONLY_51 coordinates to the nine 53R2 calls.'
elif single:
    diagnosis='BIDIRECTIONAL_SINGLE_OWNER_REGIONAL_ASSIGNMENT_DIFFERENCE'; correction='Reproduce 51R2 regional assignment exactly; the nine calls contain both additions and omissions.'
elif zero:
    diagnosis='DIFFERING_COORDINATES_HAVE_NO_NOTEBOOK45_OWNER'; correction='Resolve Notebook45 lineage before changing 53R2.'
else:
    diagnosis='REGIONAL_OWNERSHIP_AMBIGUITY_REMAINS'; correction='Resolve multi-owner regional assignment before changing 53R2.'

verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'design':'direct-18-mask-no-full-capture-index','previous_comparator':str(PREV),'source_signature_table':str(sig_path),'staged_masks':18,'shared_signatures':int(sum(common.values())),'exceptional_51':len(EL),'exceptional_53':len(ER),'pairs':len(P),'only_51_coordinates':int((C.direction=='ONLY_51').sum()),'only_53_coordinates':int((C.direction=='ONLY_53').sum()),'all_single_owner':single,'zero_owner':zero,'multi_owner':multi,'notebook45':str(N45),'diagnosis':diagnosis,'first_correction':correction,'full_capture_directories_indexed':False,'telea_rerun':False,'mask_regeneration':False,'stage_f_pass_condition':'Correct 53R2, rerun 53R2 and Notebook54, require zero Stage F pixels.'}
(OUT/'direct_18_mask_verdict.json').write_text(json.dumps(verdict,indent=2))
print(json.dumps(verdict,indent=2)); display(P); display(O)
print('COMPLETE:',OUT)
